[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/00_eda.ipynb)

# 00 — Exploratory Data Analysis

**Purpose.** Understand the inputs well enough to specify notebook 01: what the
MDT columns mean, how UEs are distributed in space and time, how the cells are
configured, and which values are impossible. PROJECT.md section 19 Step 1.

**Inputs.** `data/raw/` and `data/external/` — read-only, never modified.

**Outputs.** *Insights, not artifacts.* Nothing here writes to `data/processed/`
or `models/`. The deliverable is section 12: a written specification of what
notebook 01 must do.

**Rules.**
- `data/raw/` is never modified.
- Findings here are hypotheses. Any threshold you *derive* from this data is a
  statistical threshold and does not belong in notebook 01.
- A bound may only go into `configs/data.yaml` if you can name its external
  source — a standard, a specification, a physical limit. Record candidates in
  section 11 together with that source.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the repository? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the raw inputs

Two inputs, two shapes. The MDT export is one row per measurement; the cell
configuration is one row per cell describing the network as deployed. They are
loaded separately and joined explicitly later, in cleaning, where the join can be
audited.

In [ ]:
from src.data.load import load_cell_config, load_mdt

mdt = load_mdt(cfg)
cells = load_cell_config(cfg)

print(f"MDT records: {len(mdt):,}   cells: {len(cells)}")
mdt.head()

## 3. MDT schema and dtypes

Compare what arrived against what `configs/data.yaml` declares. A column that is
present but undeclared is a decision waiting to be made; a column that is
declared but absent is a broken contract.

In [ ]:
mdt.info()

declared = set(cfg.data.schema.mdt.columns)
observed = set(mdt.columns)
print("declared but absent:", sorted(declared - observed))
print("present but undeclared:", sorted(observed - declared))

## 4. Missingness and invalid values

Two different problems. A missing value is absent; an invalid value is present
and wrong, which is worse because nothing downstream flags it.

`find_violations` reports against the declared hard bounds, so this section sizes
the cleaning job before any record is dropped.

In [ ]:
from src.data import schema

print(mdt.isna().mean().sort_values(ascending=False))

violations = schema.find_violations(mdt, cfg, contract="mdt")
violations.groupby(["column", "rule"]).size()

## 5. RSRP distribution

The raw export reaches a maximum of `0.0` dBm, which no receiver reports — the
3GPP TS 38.133 range is `[-156, -31]`. Establish here whether that is a sentinel
for "no report" or something else, because the answer decides whether those
records are dropped or re-interpreted.

Also check the *lower* tail against the hole threshold in `configs/kpi.yaml`. If
measured RSRP never approaches -120 dBm, the MDT export is already filtered and
the measured distribution cannot be compared directly against a simulated radio
map.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(mdt["rsrp"], bins=60, ax=axes[0])
axes[0].axvline(cfg.kpi.hole_dbm, ls="--", label="hole")
axes[0].axvline(cfg.kpi.weak_dbm, ls=":", label="weak")
axes[0].legend()
axes[0].set_title("RSRP distribution")

sentinel = mdt["rsrp"] >= cfg.data.schema.mdt.columns.rsrp.max
print(f"records above the reportable maximum: {sentinel.sum():,} ({sentinel.mean():.2%})")
mdt["rsrp"].describe()

## 6. UE spatial distribution

This becomes `rho(g)`, the weight in the UE-weighted Band Priority Score
(PROJECT.md section 7). Two things to establish:

1. **Extent.** How far do the measurements reach, and does that match the scene?
   Records outside the scene have no grid cell to contribute to.
2. **Concentration.** How uneven is the density? The Band Priority Score is a
   UE-weighted average, so a handful of very dense cells can dominate it — which
   is worth knowing before the score is used as an objective.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(mdt["sim_x"], mdt["sim_y"], s=1, alpha=0.2, label="MDT")
ax.scatter(cells["sim_x"], cells["sim_y"], marker="^", s=80, label="cells")
ax.set_aspect("equal")
ax.legend()
ax.set_title("UE observations and cell sites")

print(mdt[["sim_x", "sim_y"]].describe())
# TODO: how much of the extent is empty? Compare the MDT bounding box with the
# convex hull of the cell sites — a large gap means most of the grid carries no
# UE weight at all.

## 7. Temporal distribution

Establish the observation window and whether it is continuous. This drives the
`temporal` split option in `configs/data.yaml`: a window with a gap in the middle
splits very differently from a uniform one.

Also check whether the network configuration changed during the window. The cell
configuration carries a single `sync_date`, so if measurements predate it, they
were taken against a network that is not the one being modelled.

In [ ]:
print("MDT window:", mdt["date"].min(), "->", mdt["date"].max())
print("cell config sync_date:", sorted(cells["sync_date"].unique()))

mdt.set_index("date").resample("1D").size().plot(figsize=(11, 3), title="records per day")
# TODO: do any measurements predate the cell configuration sync_date? Those were
# taken against a different network and cannot be compared to this scene.

## 8. Cell configuration

The network as deployed: how many sites, how many sectors each, and how the
azimuths are arranged. This determines the MARL agent granularity choice in
`configs/optim/marl.yaml` — `cell` or `site` — so it is worth understanding
before that decision is made.

Check the tilt values too. They are the baseline every reported improvement is
measured against.

In [ ]:
print(cells.groupby("gnodeb_id").size().describe())
print("\ntilt values:", sorted(cells["digital_tilt"].unique()))
print("antenna heights:", sorted(cells["antenna_height"].unique()))

fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(5, 5))
ax.scatter(np.deg2rad(cells["azimuth"]), np.ones(len(cells)))
ax.set_title("cell azimuths")

## 9. The band dimension

**This is the gap that blocks the pipeline.** PROJECT.md optimises one absolute
tilt per `(cell, band)` pair, and PROJECT.md section 4.2 requires a carrier
frequency, a transmit power, and an `eTilt`/`mTilt` split per cell-band. The
current export has none of them — one `digital_tilt` per cell and no band column.

Record here exactly what is missing and what the multi-band export will need to
carry, so `configs/data.yaml` and `configs/radio.yaml` can be filled in the
moment it arrives. See docs/adr/0005.

In [ ]:
pending = [c for c in cfg.data.schema.cell_config.columns if c.startswith("<")]
print("columns declared but not yet exported:")
for c in pending:
    print(" ", c)

# TODO: confirm with the data owner —
#   1. how many bands per cell, and which?
#   2. is digital_tilt the electrical tilt, and is mechanical tilt separate?
#   3. what are the settable tilt bounds per band, and the RET step size?
#   4. transmit power per cell-band.

## 10. Scene extent and coordinate alignment

MDT positions, cell positions and the Sionna-RT scene must share one coordinate
frame. Check it here rather than discovering a mismatch after a ray-tracing run:
if the cells do not sit inside the scene, every radio map will be empty and the
KPIs will report a network that is entirely a coverage hole.

In [ ]:
from src.radio.scene import scene_bounds

xmin, ymin, xmax, ymax = scene_bounds(cfg)
print(f"scene:  x [{xmin}, {xmax}]   y [{ymin}, {ymax}]")
print(f"cells:  x [{cells.sim_x.min()}, {cells.sim_x.max()}]")
print(f"MDT:    x [{mdt.sim_x.min()}, {mdt.sim_x.max()}]")

outside = ~mdt["sim_x"].between(xmin, xmax) | ~mdt["sim_y"].between(ymin, ymax)
print(f"MDT records outside the scene: {outside.sum():,} ({outside.mean():.2%})")

## 11. Candidate hard constraints

Every rule that notebook 01 will apply, together with the external source that
authorises it. A rule with no source is a statistical filter and does not belong
in cleaning.

| Rule | Source | Records affected |
|---|---|---|
| `rsrp` in `[-156, -31]` dBm | 3GPP TS 38.133 reporting range | *fill in from section 5* |
| `gcell_id` present in the cell configuration | join integrity | *fill in from section 4* |
| position inside the scene | scene extent, section 10 | *fill in from section 10* |
| exact duplicate records | — | *fill in from section 4* |

In [ ]:
# TODO: quantify each candidate rule above and paste the counts into the table.
# The point of the table is that notebook 01 drops a known number of records for
# a stated reason — not an unknown number for a plausible one.

## 12. Findings — what 01 and 02 must do

*Written by you, from the sections above. Replace every placeholder.*

**Cleaning rules for notebook 01.** *Which rules, with the source and expected
record count for each.*

**Split method.** *`spatial_block`, `temporal` or `group_shuffle`, and why — what
kind of generalisation does the reported result need to demonstrate?*

**Grid resolution.** *A value for `cfg.radio.grid.cell_size_m`, justified against
the UE density in section 6 and the ray-tracing cost.*

**Band parameters still needed.** *The list from section 9, and who is supplying
them.*

**Anything that changes the formulation.** *If the data contradicts PROJECT.md —
for example if MDT RSRP is pre-filtered above the hole threshold — say so here.
That is an ADR, not a notebook comment.*